# DAY 5: Rigorous Evaluation: Base Model vs. Fine-Tuned Model

## 🎯 Learning Objectives
Never settle for "it looks better". In production LLM engineering, you must prove improvement quantitatively.

By the end of this day, you will be able to execute and explain:
1. **Held-Out Test Benchmarking**: Evaluating the Base Model and Fine-Tuned Model against the exact same unseen test split.
2. **Structured Evaluation Metrics**:
   - **JSON Validity Rate**: Proportion of outputs that successfully parse into valid JSON.
   - **Schema Conformance Rate**: Presence and type-correctness of all 5 schema keys.
   - **Exact Match Rate**: Strict equality across all extracted fields simultaneously.
   - **Field-Level Accuracy**: Accuracy for `customer`, `quantity`, `product`, `amount`, `delivery_day`.
3. **Inference Latency & Token Efficiency**: Comparing response length and generation speed.
4. **Failure Mode Analysis**: Diagnosing why base models fail (hallucination, markdown wrapping, preamble chat) vs. how fine-tuning enforces strict formatting.

In [ ]:
# !pip install -q transformers peft accelerate datasets pyyaml

In [ ]:
import json
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from src.evaluation.metrics import evaluate_predictions
from src.data.formatter import format_chat_prompt, apply_template_to_messages

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "models/adapters/qwen-1.5b-order-extractor"
TEST_FILE = "data/processed/test.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Device: {device} ({torch_dtype})")

### Step 1: Load Held-Out Test Set

In [ ]:
test_samples = []
with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_samples.append(json.loads(line))

print(f"Loaded {len(test_samples)} held-out test samples from {TEST_FILE}.")
print(f"First test input: {test_samples[0]['input']}")
print(f"First target: {test_samples[0]['output']}")

### Step 2: Benchmark Base Model (Zero-Shot)

In [ ]:
print(f"Loading Base Model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None
)
base_model.eval()

base_predictions = []
ground_truths = [s["output"] for s in test_samples]

start_t = time.time()
for idx, sample in enumerate(test_samples):
    msgs = format_chat_prompt(sample["input"])
    prompt_text = apply_template_to_messages(msgs, tokenizer, add_generation_prompt=True)
    inp = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = base_model.generate(**inp, max_new_tokens=100, do_sample=False)
    
    gen_text = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    base_predictions.append(gen_text)

base_time = time.time() - start_t
base_metrics, base_details = evaluate_predictions(base_predictions, ground_truths)
print("=== BASE MODEL METRICS ===")
print(f"JSON Validity Rate : {base_metrics['json_validity_rate'] * 100:.2f}%")
print(f"Exact Match Rate   : {base_metrics['exact_match_rate'] * 100:.2f}%")
print(f"Field Accuracies   : {base_metrics['field_accuracies']}")

### Step 3: Benchmark Fine-Tuned Model (with Adapter)

In [ ]:
print(f"Attaching Fine-Tuned Adapter from {ADAPTER_PATH}...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()

ft_predictions = []
start_t = time.time()
for idx, sample in enumerate(test_samples):
    msgs = format_chat_prompt(sample["input"])
    prompt_text = apply_template_to_messages(msgs, tokenizer, add_generation_prompt=True)
    inp = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = ft_model.generate(**inp, max_new_tokens=100, do_sample=False)
    
    gen_text = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    ft_predictions.append(gen_text)

ft_time = time.time() - start_t
ft_metrics, ft_details = evaluate_predictions(ft_predictions, ground_truths)
print("=== FINE-TUNED MODEL METRICS ===")
print(f"JSON Validity Rate : {ft_metrics['json_validity_rate'] * 100:.2f}%")
print(f"Exact Match Rate   : {ft_metrics['exact_match_rate'] * 100:.2f}%")
print(f"Field Accuracies   : {ft_metrics['field_accuracies']}")

### Step 4: Comparison Table & Report Generation

In [ ]:
print("=" * 70)
print(f"{'Metric':<28} | {'Base Model':<16} | {'Fine-Tuned Model':<16}")
print("=" * 70)
print(f"{'JSON Validity Rate':<28} | {base_metrics['json_validity_rate']*100:>14.2f}% | {ft_metrics['json_validity_rate']*100:>14.2f}%")
print(f"{'Exact Match Rate':<28} | {base_metrics['exact_match_rate']*100:>14.2f}% | {ft_metrics['exact_match_rate']*100:>14.2f}%")
print(f"{'Avg Field Accuracy':<28} | {base_metrics['avg_field_accuracy']*100:>14.2f}% | {ft_metrics['avg_field_accuracy']*100:>14.2f}%")
for k in base_metrics['field_accuracies']:
    print(f"  - {k:<24} | {base_metrics['field_accuracies'][k]*100:>14.2f}% | {ft_metrics['field_accuracies'][k]*100:>14.2f}%")
print("=" * 70)

---
## 🎓 Day 5 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Exact Match vs Field-Level Precision/Recall**: In structured tasks, exact match is the strictest metric (all fields correct). Field-level accuracy reveals granular entity extraction performance.
2. **Greedy Decoding for Evaluation**: Always use deterministic greedy decoding (`temperature=0.0`, `do_sample=False`) during benchmark evaluations to ensure reproducibility.
3. **Preamble Hallucination Elimination**: Fine-tuning teaches the model to immediately output `{` and close with `}`, omitting conversational preambles that break downstream JSON parsers.
4. **LLM-as-a-Judge**: Using a larger model (e.g. GPT-4) to evaluate free-form text. For structured JSON with known ground truth, rule-based schema validators are strictly superior and deterministic.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *Why is Perplexity or Training Loss insufficient to prove your fine-tuned model works in production?*
   - **Answer**: Perplexity only measures next-token probability prediction. A model can have low perplexity while still outputting malformed JSON or omitting required fields. Downstream task metrics (JSON validity, Schema Conformance, Field Accuracy) are the true test of functional utility.
2. **Q2 (Intermediate)**: *Why is temperature set to 0.0 during evaluation?*
   - **Answer**: Setting temperature to 0.0 forces the model to select the argmax (highest probability) token at each step, removing sampling variance and ensuring exact reproducibility across benchmark runs.
3. **Q3 (Intermediate)**: *What was the most common failure mode of the base model before fine-tuning?*
   - **Answer**: Chattiness (e.g., adding "Here is your JSON:" or markdown backticks ```json) and occasional field naming variations (e.g., "total" instead of "amount").
4. **Q4 (Advanced)**: *How do you evaluate structured JSON extraction when field values can have synonyms (e.g. "laptops" vs "laptop computers")?*
   - **Answer**: By applying canonicalization functions during evaluation (e.g. lowercasing, stripping punctuation, or mapping synonyms to standard ontology dictionaries) or evaluating string semantic similarity via embeddings.
5. **Q5 (Advanced)**: *How would you set up automated continuous evaluation (CI/CD) for model fine-tuning?*
   - **Answer**: Run an automated evaluation pipeline in GitHub Actions or model registry webhooks that runs the model on the gold test set upon PR, blocking deployment if Exact Match Rate or JSON Validity falls below established SLA thresholds.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, how did you calculate field-level accuracy across the 5 schema keys?*
2. *How did `evaluate_single_sample()` handle cases where the raw model string was not valid JSON?*
3. *What was the exact percentage improvement in Exact Match Rate achieved by fine-tuning?*

### 🏆 Day 5 Hands-On Challenge
> **Challenge**: Implement an error breakdown function that groups failed test samples by error category: `SYNTAX_ERROR`, `MISSING_FIELD`, `VALUE_MISMATCH`, or `TYPE_ERROR`.